# 12. Reference-first Feedback Judge and Measurements

Extract one fixed concern inventory per reflection, review and approve it, then judge saved G1/G2/G3 feedback against it. Extract packet-level measurements and a separate concern audit locally.

Judge `0.12.0` measures reference-relative discrepancies, not independently established pedagogical quality. Human experts assess pedagogical quality separately. This notebook is configured for remaining-test: 18 reflections, one inventory per reflection and one judgment per G1/G2/G3 output. Provider execution switches default to false; automatic inventory finalization is authorized and configured separately. No generation or assessment-quality calls are made, and the earlier assessment study remains paused.

Notebook outputs may contain protected student and lecturer text. Clear all outputs before committing this notebook.

In [5]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

from reflection_assessment_feedback.experiment_config import PROJECT_ROOT, load_experiment_config
from reflection_assessment_feedback.judge_artifacts import content_hash, private_directory, protected_path, write_once
from reflection_assessment_feedback.reference_inventory import (
    ExtractorConfig, approve_inventory, extract_reference_inventory,
    inventory_reference, load_inventory_artifact, reference_input_from_packet,
)
from reflection_assessment_feedback.judge_study import (
    freeze_judge_study, load_judge_study, run_judge_study, study_progress,
)
from reflection_assessment_feedback.judge_stability import (
    analyze_judge_study, extract_feedback_measurements, summarize_feedback_metrics,
)

load_dotenv(PROJECT_ROOT / '.env', override=False)
if not os.environ.get('PREBI_DATA_ROOT'):
    raise RuntimeError('Set PREBI_DATA_ROOT to approved protected storage.')
PROTECTED_ROOT = protected_path(os.environ['PREBI_DATA_ROOT'])
INVENTORY_ROOT = PROTECTED_ROOT / 'llm-judge' / 'reference-inventories'
judge_config = load_experiment_config()['llm_judge']

## 1. Select inputs and actions

The remaining-test document IDs are read from the existing frozen assessment study, used only as a reference-input source. No assessment execution is resumed. Inventories start empty and are populated by cached extraction. Feedback packets are selected from saved remaining-test generations (`FEEDBACK_SOURCE_STUDY=None`); freezing checks that their reflection, rubric and lecturer feedback match the approved inventories. Expect up to 18 extraction calls plus 54 judge calls, before cache/resume savings. No feedback regeneration.

To resume or analyze, set `STUDY_DIRECTORY` to an existing feedback study and leave `FREEZE_STUDY=False`. Its frozen protocol controls execution, not the current selection settings. The two existing v0.12.0 playground study IDs are `79a0529c5f303e38825ec691e46f817ead0194437b0c633f2ce54f757633318c` (3/126) and `a5656883504b3386383ede8a81be6eec2358e88efe23dbb624ed0b48bcca2ad8` (64/188), under protected `llm-judge/stability-studies/`.

In [6]:
COHORT = 'remaining-test'
SOURCE_STUDY = PROTECTED_ROOT / 'llm-judge' / 'stability-studies' / '28b89a148dde0590684639ef2b1a49ea532f0f9e9ba8b9f883e97205ea747cf3'
source_protocol = load_judge_study(SOURCE_STUDY)['protocol']
if source_protocol['cohort'] != COHORT:
    raise ValueError('Reference source must belong to remaining-test.')
DOCUMENT_IDS = tuple(sorted({entry['identity']['document_id'] for entry in source_protocol['entries']}))
CONDITIONS = ('G1', 'G2', 'G3')
JUDGE_REPETITIONS = 1
EXTRACTOR_PROMPT_VERSION = '0.2.0'
JUDGE_PROMPT_VERSION = '0.12.0'
EXTRACTION_REVISION = '1'
FEEDBACK_SOURCE_STUDY = None
STUDY_DIRECTORY = None

INVENTORY_IDS = {}
EXTRACT_INVENTORIES = False
APPROVE_INVENTORIES = False
AUTO_FINALIZE_INVENTORIES = True
APPROVED_BY = 'quynguyen (explicit remaining-test automatic finalization authorization)'
FREEZE_STUDY = True
RUN_JUDGE = True
APPROVE_EXTERNAL_PROCESSING = False
APPROVE_FULL_BATCH = False
EXPORT_TABLES = False
EXPORT_FILENAME = 'remaining-test-feedback-measurements-v1.json'
print('Selected reflections:', len(DOCUMENT_IDS), DOCUMENT_IDS)
print('Maximum uncached extraction calls:', len(DOCUMENT_IDS))
print('Planned judge calls:', len(DOCUMENT_IDS) * len(CONDITIONS) * JUDGE_REPETITIONS)
print('To execute: enable EXTRACT_INVENTORIES, FREEZE_STUDY, RUN_JUDGE and both processing/batch approvals. Keep assessment study paused.')

Selected reflections: 18 ('102', '110', '113', '158', '161', '164', '172', '174', '224', '231', '233', '36', '39', '42', '49', '51', '96', '99')
Maximum uncached extraction calls: 18
Planned judge calls: 54
To execute: enable EXTRACT_INVENTORIES, FREEZE_STUDY, RUN_JUDGE and both processing/batch approvals. Keep assessment study paused.


## 2. Optional reference-only extraction

Enable `EXTRACT_INVENTORIES` and external-processing approval only when needed. This sends reflection segments, rubric and lecturer feedback, never generated feedback or generator condition. Each source is extracted once, not per generator or repetition. Exact cached configurations reuse their inventory. Changing source, prompt or extraction revision creates a new immutable inventory. Approved EU tracing follows the experiment configuration. Extraction failures stop explicitly.

In [7]:
if EXTRACT_INVENTORIES:
    if not APPROVE_EXTERNAL_PROCESSING or SOURCE_STUDY is None:
        raise RuntimeError('Extraction requires external-processing approval and SOURCE_STUDY.')
    if AUTO_FINALIZE_INVENTORIES and not APPROVED_BY.strip():
        raise RuntimeError('Automatic finalization requires APPROVED_BY before extraction.')
    source_protocol = load_judge_study(protected_path(SOURCE_STUDY))['protocol']
    if source_protocol['cohort'] != COHORT:
        raise ValueError('Source cohort differs from selected cohort.')
    sources = {}
    for entry in source_protocol['entries']:
        document = entry['identity']['document_id']
        if document not in DOCUMENT_IDS:
            continue
        source = reference_input_from_packet(entry['packet'], document)
        if document in sources and content_hash(sources[document]) != content_hash(source):
            raise ValueError(f'Conflicting reference-only inputs for {document}.')
        sources[document] = source
    if set(sources) != set(DOCUMENT_IDS):
        raise ValueError('Selected documents are missing from SOURCE_STUDY.')
    extraction_config = ExtractorConfig(
        provider=judge_config['provider'], model_name=judge_config['model_name'],
        prompt_version=EXTRACTOR_PROMPT_VERSION, extraction_revision=EXTRACTION_REVISION,
    )
    for document, source in sources.items():
        directory = extract_reference_inventory(
            PROTECTED_ROOT, source=source, config=extraction_config,
            approve_external_processing=True,
            enable_langsmith_tracing=judge_config['enable_langsmith_tracing'],
            minimum_request_interval_seconds=judge_config['minimum_request_interval_seconds'],
        )
        INVENTORY_IDS[document] = directory.name
        if AUTO_FINALIZE_INVENTORIES:
            approve_inventory(directory, approved_by=APPROVED_BY)
        print(document, directory)
else:
    print('Extraction disabled; no extractor calls.')

Extraction disabled; no extractor calls.


## 3. Review inventories, then explicitly approve

Automatic finalization immediately after each validated extraction is enabled under the user's explicit remaining-test authorization. The table remains available for auditing exact excerpts, boundaries, support and evaluable scopes. Approval is write-once and records this authorization; never edit finalized inventories. To require manual review instead, set AUTO_FINALIZE_INVENTORIES=False, inspect this table, then enable APPROVE_INVENTORIES. Missing context and uncertain support are independent.

In [8]:
inventory_records = {}
for document in DOCUMENT_IDS:
    if document not in INVENTORY_IDS:
        raise ValueError(f'Select or extract an inventory for {document}.')
    record = load_inventory_artifact(INVENTORY_ROOT / INVENTORY_IDS[document])
    if record['inventory']['reflection_id'] != document:
        raise ValueError('Inventory reflection identity mismatch.')
    inventory_records[document] = record
    print('Reflection:', document, 'Inventory:', record['inventory_id'])
    display(pd.DataFrame(record['inventory']['reference_concerns']))

ValueError: Select or extract an inventory for 102.

In [ ]:
if APPROVE_INVENTORIES and not AUTO_FINALIZE_INVENTORIES:
    if not APPROVED_BY.strip():
        raise RuntimeError('Set APPROVED_BY after reviewing the inventories.')
    for document in DOCUMENT_IDS:
        print(approve_inventory(
            INVENTORY_ROOT / INVENTORY_IDS[document], approved_by=APPROVED_BY,
        ))
else:
    print('Inventory approval disabled. Existing approvals remain unchanged.')

## 4. Freeze locally or load a saved study

Freezing verifies approvals and reference-source consistency, and embeds each inventory once. It makes no LLM calls. Existing remaining-test feedback is reused: 18 reflections x three generators x one judge repetition = 54 judgments. Previous assessment or playground judgments are not imported. Inspect actual pending-call counts before execution. To resume after restarting the kernel, re-enable extraction to recover identical cached inventories without new calls, then repeat the same freeze, or set STUDY_DIRECTORY to the printed saved path and keep FREEZE_STUDY=False.

In [ ]:
if FREEZE_STUDY:
    if STUDY_DIRECTORY is not None:
        raise ValueError('Choose either a new freeze or an existing STUDY_DIRECTORY.')
    references = {}
    for document in DOCUMENT_IDS:
        record = load_inventory_artifact(
            INVENTORY_ROOT / INVENTORY_IDS[document], require_approved=True,
        )
        references[document] = inventory_reference(record)
    STUDY_DIRECTORY = freeze_judge_study(
        PROTECTED_ROOT, cohort=COHORT, conditions=CONDITIONS,
        tasks=('feedback_quality',), judge_repetitions=JUDGE_REPETITIONS,
        import_existing=False, feedback_prompt_version=JUDGE_PROMPT_VERSION,
        document_ids=DOCUMENT_IDS, reference_inventory_refs=references,
        source_study=FEEDBACK_SOURCE_STUDY,
    )
if STUDY_DIRECTORY is None:
    raise RuntimeError('Set FREEZE_STUDY=True or select an existing STUDY_DIRECTORY.')
STUDY_DIRECTORY = protected_path(STUDY_DIRECTORY)
frozen = load_judge_study(STUDY_DIRECTORY)
protocol = frozen['protocol']
if {entry['packet']['task'] for entry in protocol['entries']} != {'feedback_quality'} or protocol.get('feedback_prompt_version') != JUDGE_PROMPT_VERSION:
    raise ValueError('Selected study must be feedback-only with the selected judge version.')
print('Study:', STUDY_DIRECTORY)
print(study_progress(STUDY_DIRECTORY))
display(pd.DataFrame([
    {'document': entry['identity']['document_id'], 'condition': entry['identity']['condition'],
     'packet_id': entry['packet']['packet_id'],
     'inventory_id': entry['packet']['reference_inventory_ref']['inventory_id']}
    for entry in protocol['entries']
]))

## 5. Execute only after reviewing the frozen call projection

Enable `RUN_JUDGE`, `APPROVE_EXTERNAL_PROCESSING` and `APPROVE_FULL_BATCH` explicitly. One call per unsaved packet/repetition; inventories are not re-extracted. The frozen study governs execution and tracing. Failures stop the run; inspect them before authorizing a replacement. A rerun retries the missing slot and can incur extra provider cost. Completed results are write-once and skipped on resume.

In [ ]:
if RUN_JUDGE:
    if not (APPROVE_EXTERNAL_PROCESSING and APPROVE_FULL_BATCH):
        raise RuntimeError('Both processing and full-batch approvals are required.')
    print(run_judge_study(
        STUDY_DIRECTORY, approve_external_processing=True, approve_full_batch=True,
    ))
else:
    print('Judging disabled; saved results can still be analyzed locally.')

## 6. Local measurements and concern audit

One measurement row per packet **and judge repetition**; no pooling. Central actionable concerns are central critical/developmental concerns with supported/uncertain support and at least partially evaluable scope. Positive, secondary, unsupported and non-evaluable concerns are excluded. `central_actionable_n` includes eligible `unable_to_judge` concerns, but rate denominators exclude them. Partial coverage receives no fractional credit. Zero-denominator rates and unsaved outcome measurements are null, not zero. Issue flags use actual Issues independently of coverage or grouping.

The separate concern table retains all concerns, including exclusions, with frozen definitions/scopes, evidence, denominator flags and Issue links. Missing coverage does not automatically mean a pedagogical error or substantive Issue.

In [ ]:
tables = extract_feedback_measurements(STUDY_DIRECTORY)
measurements = tables['measurements']
concern_audit = tables['concern_coverage']
analysis = analyze_judge_study(STUDY_DIRECTORY)
display(measurements)
metric_summary = summarize_feedback_metrics(measurements)
display(metric_summary)
display(concern_audit)
display(analysis['issues'])
print('Measurement rows:', len(measurements), 'Concern audit rows:', len(concern_audit))

## Playground metrics across all four reflections

Run setup first. This local-only cell loads the two completed v0.12.0 playground studies directly, independent of STUDY_DIRECTORY and execution switches. It displays packet-level rates and equal-weight (macro) means by generator across reflections, not pooled concern counts. PRR (higher is better) is addressed / eligible positive concerns, regardless of centrality. CMR (lower is better) is missing / eligible central actionable concerns. CFCR (higher is better) is addressed / eligible central actionable concerns. Major Discrepancy Incidence (lower is better) is feedbacks with at least one major Issue / evaluated feedbacks. Positive concerns use the same support/evaluability exclusions; unable_to_judge is excluded from concern-rate denominators. Partial coverage receives no numerator credit. Undefined rates remain missing; available-rate counts and actual denominators are shown. Repetitions remain separate. The full audit is retained as playground_concern_audit. No LLM calls or file writes. Rerun this cell to replace earlier saved metric displays.

In [ ]:
playground_study_ids = (
    '79a0529c5f303e38825ec691e46f817ead0194437b0c633f2ce54f757633318c',
    'a5656883504b3386383ede8a81be6eec2358e88efe23dbb624ed0b48bcca2ad8',
)
playground_tables = []
for study_id in playground_study_ids:
    directory = PROTECTED_ROOT / 'llm-judge' / 'stability-studies' / study_id
    saved_protocol = load_judge_study(directory)['protocol']
    if saved_protocol['cohort'] != 'playground' or saved_protocol.get('feedback_prompt_version') != '0.12.0':
        raise ValueError('Playground metrics require playground studies with judge 0.12.0.')
    playground_tables.append(extract_feedback_measurements(directory))
playground_measurements = pd.concat([t['measurements'] for t in playground_tables], ignore_index=True)
playground_concern_audit = pd.concat([t['concern_coverage'] for t in playground_tables], ignore_index=True)
metric_keys = ['reflection_id', 'packet_id', 'condition', 'generation_repetition', 'judge_repetition']
if playground_measurements.duplicated(['packet_id', 'judge_repetition']).any():
    raise ValueError('Overlapping packet/repetition results must not be pooled.')
if set(playground_measurements['reflection_id']) != {'3', '64', '126', '188'} or len(playground_measurements) != 12:
    raise ValueError('Expected exactly four playground reflections x three generators x one repetition.')
if (playground_measurements['overall_severity'] == 'missing').any():
    raise ValueError('Playground study has unfinished judgments.')
coverage_denominators = playground_concern_audit.groupby(metric_keys)['in_coverage_rate_denominator'].sum().rename('coverage_denominator_n')
playground_packet_metrics = playground_measurements.merge(coverage_denominators, on=metric_keys, validate='one_to_one')
playground_metrics_by_condition = summarize_feedback_metrics(playground_packet_metrics)
print('Per-packet measurements (rates are proportions, not percentages):')
display(playground_packet_metrics.sort_values(['reflection_id', 'condition']))
print('Equal-weight means across the four reflections; descriptive, not a stable generator ranking:')
display(playground_metrics_by_condition)

## Remaining-test feedback metrics (G1/G2/G3, micro-averaged)

Run setup first. This local-only cell reads saved judgments; it makes no LLM calls and writes no files. G1 and G3 come from the combined remaining-test study, and G2 from the author-disjoint G2 study (generation prompt 2.0.0); the combined study's earlier G2 is excluded. The cell checks judge `0.12.0`, complete judgments, the same reflections for every generator and one frozen inventory per reflection.

Counts are summed across reflections before division (micro-averaging); percentages are rounded to two decimals, with raw counts shown separately. PRR = fully addressed / eligible positive concerns. CMR, CPCR and CFCR = missing, partially addressed or fully addressed / eligible central critical/developmental concerns; together they form one coverage distribution summing to 100%, not three independent measures. MDI = feedbacks with at least one judge-classified major Issue / evaluated feedbacks; missing concerns are not counted as discrepancies. Non-evaluable, unsupported and `unable_to_judge` concerns are excluded, zero denominators are null, and repetitions are not pooled.

In [9]:
from reflection_assessment_feedback.judge_stability import summarize_micro_feedback_metrics

remaining_metric_sources = {
    # study_id: generators taken from that study
    'a31a8ec016cedcc21939be1fc249e940b20ad8911a31934be7dd724a5176f6c3': ('G1', 'G3'),
    '66128161ef618afbaea11928e5abf040cb3df103e8837fda5b229e790b1656aa': ('G2',),
}
remaining_tables, remaining_inventory_refs = [], {}
for study_id, generators in remaining_metric_sources.items():
    directory = PROTECTED_ROOT / 'llm-judge' / 'stability-studies' / study_id
    saved_protocol = load_judge_study(directory)['protocol']
    if saved_protocol['cohort'] != 'remaining-test' or saved_protocol.get('feedback_prompt_version') != '0.12.0':
        raise ValueError(f'{study_id[:12]}: expected remaining-test study with judge 0.12.0.')
    progress = study_progress(directory)
    if progress['saved_judgments'] != progress['planned_judgments']:
        raise ValueError(f'{study_id[:12]}: unfinished judgments; complete the judge run first.')
    for saved_entry in saved_protocol['entries']:
        if saved_entry['identity']['condition'] in generators:
            reference = saved_entry['packet']['reference_inventory_ref']
            remaining_inventory_refs.setdefault(reference['reflection_id'], set()).add(
                (reference['inventory_id'], reference['inventory_sha256']),
            )
    study_tables = extract_feedback_measurements(directory)
    remaining_tables.append({
        name: table[table['condition'].isin(generators)] for name, table in study_tables.items()
    })
remaining_measurements = pd.concat([t['measurements'] for t in remaining_tables], ignore_index=True)
remaining_concern_audit = pd.concat([t['concern_coverage'] for t in remaining_tables], ignore_index=True)
if any(len(refs) != 1 for refs in remaining_inventory_refs.values()):
    raise ValueError('Generators were judged against different inventories for the same reflection.')
reflections_by_generator = remaining_measurements.groupby('condition')['reflection_id'].agg(frozenset)
if set(reflections_by_generator.index) != {'G1', 'G2', 'G3'} or reflections_by_generator.nunique() != 1:
    raise ValueError('G1, G2 and G3 must cover the same reflections.')
if (remaining_measurements['overall_severity'] == 'missing').any():
    raise ValueError('Unexecuted judgments found.')

remaining_micro_metrics = summarize_micro_feedback_metrics(remaining_measurements)
print('Reflections per generator:', len(reflections_by_generator.iloc[0]))
display(remaining_micro_metrics[['Generator', 'generation_repetition', 'judge_repetition', 'PRR', 'CMR', 'CPCR', 'CFCR', 'MDI']])
display(remaining_micro_metrics.drop(columns=['PRR', 'CMR', 'CPCR', 'CFCR', 'MDI']))

Reflections per generator: 18


,Generator,generation_repetition,judge_repetition,PRR,CMR,CPCR,CFCR,MDI
0,G1,1,1,76.12,62.5,20.83,16.67,72.22
1,G2,1,1,46.27,54.17,16.67,29.17,77.78
2,G3,1,1,56.72,47.92,18.75,33.33,72.22


,Generator,generation_repetition,judge_repetition,positive_addressed_n,positive_denominator_n,central_missing_n,central_partial_n,central_addressed_n,central_denominator_n,major_discrepancy_feedbacks_n,evaluated_feedbacks_n
0,G1,1,1,51,67,30,10,8,48,13,18
1,G2,1,1,31,67,26,8,14,48,14,18
2,G3,1,1,38,67,23,9,16,48,13,18


## 7. Optional protected export

Judgments are already persisted automatically. This optional JSON export includes measurements, the concern audit, Issues and methodology/provenance. It makes no provider calls and never overwrites an existing file. Choose a new filename for a later snapshot. Exports remain inside the protected study directory, never in the source repository.

In [ ]:
if EXPORT_TABLES:
    if Path(EXPORT_FILENAME).name != EXPORT_FILENAME or not EXPORT_FILENAME.endswith('.json'):
        raise ValueError('EXPORT_FILENAME must be a plain JSON filename.')
    if not STUDY_DIRECTORY.is_relative_to(PROTECTED_ROOT):
        raise ValueError('Export study must be inside PREBI_DATA_ROOT.')
    export_directory = STUDY_DIRECTORY / 'measurement-exports'
    private_directory(export_directory, PROTECTED_ROOT)
    export = {
        'schema_version': '1.1.0', 'study_id': frozen['study_id'],
        'judge_prompt_version': protocol['feedback_prompt_version'],
        'measurement_definition': {
            'central_actionable': 'central critical/developmental; supported/uncertain; at least partially evaluable',
            'rate_denominator': 'central_actionable excluding unable_to_judge',
            'partial_credit': False,
            'positive_eligible': 'positive of any centrality; supported/uncertain; at least partially evaluable; exclude unable_to_judge from denominator',
            'aggregation': 'equal-weight reflection means; generation and judge repetitions separate',
        },
        **{name: json.loads(table.to_json(orient='records', force_ascii=False))
           for name, table in tables.items()},
        'metric_summary': json.loads(metric_summary.to_json(orient='records', force_ascii=False)),
        'issues': json.loads(analysis['issues'].to_json(orient='records', force_ascii=False)),
    }
    target = export_directory / EXPORT_FILENAME
    write_once(target, export)
    print('Saved:', target)
else:
    print('Export disabled.')